<a href="https://colab.research.google.com/github/simranshika29/data_science_capstone/blob/main/Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn gensim nltk transformers torch tqdm matplotlib seaborn spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os, re, numpy as np, pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
from transformers import BertTokenizer, BertModel
import torch
from sklearn.manifold import TSNE

nltk.download('punkt')
nltk.download('stopwords')
nlp = spacy.load('en_core_web_sm')
STOPWORDS = set(stopwords.words('english'))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


device(type='cpu')

In [4]:
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("IMDB Dataset.csv")
df = df[['review', 'sentiment']].dropna()
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1})
df.head()

Saving IMDB Dataset.csv to IMDB Dataset (1).csv


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [ ]:
def clean_text(text):
    # remove html
    text = re.sub(r"<.*?>", " ", text)
    # keep only letters and apostrophes
    text = re.sub(r"[^a-zA-Z']", " ", text)
    text = text.lower()
    # spaCy doc for lemmatization and POS-aware processing
    doc = nlp(text)
    tokens = []
    for token in doc:
        if token.is_stop:
            continue
        lemma = token.lemma_.strip()
        if len(lemma) > 1 and lemma.isalpha() and lemma not in STOPWORDS:
            tokens.append(lemma)
    return " ".join(tokens)

df['clean_review'] = df['review'].apply(clean_text)
X = df['clean_review'].values
y = df['label'].values
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
len(X_train_text), len(X_test_text)

In [ ]:
bow_vectorizer = CountVectorizer(max_features=20000)
X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow = bow_vectorizer.transform(X_test_text)
tfidf_vectorizer = TfidfVectorizer(max_features=20000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)
tfidf_ngram = TfidfVectorizer(ngram_range=(1, 2), max_features=40000)
X_train_ngram = tfidf_ngram.fit_transform(X_train_text)
X_test_ngram = tfidf_ngram.transform(X_test_text)
X_train_bow.shape, X_train_tfidf.shape, X_train_ngram.shape

In [ ]:
def train_model(X_train, X_test, y_train, y_test, name):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n===== {name} =====")
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred, digits=4))
    return clf, y_pred, acc

In [ ]:
results = {}
bow_clf, y_pred_bow, results['BoW'] = train_model(
    X_train_bow, X_test_bow, y_train, y_test, "BoW"
)
tfidf_clf, y_pred_tfidf, results['TF-IDF'] = train_model(
    X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF"
)
ngram_clf, y_pred_ngram, results['TF-IDF N-gram'] = train_model(
    X_train_ngram, X_test_ngram, y_train, y_test, "TF-IDF (Uni+Bi-gram)"
)
results

In [ ]:
def show_top_words(vectorizer, clf, top_n=20):
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = clf.coef_[0]
    top_pos_idx = np.argsort(coefs)[-top_n:]
    top_neg_idx = np.argsort(coefs)[:top_n]
    print("\nTop Positive Words:")
    print(feature_names[top_pos_idx][::-1])
    print("\nTop Negative Words:")
    print(feature_names[top_neg_idx])


show_top_words(tfidf_vectorizer, tfidf_clf, top_n=20)

In [ ]:
sentences = [s.split() for s in X_train_text]
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=5,
)
len(w2v_model.wv.index_to_key)

In [ ]:
def w2v_sentence_vec(text):
    words = text.split()
    vecs = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if not vecs:
        return np.zeros(100)
    return np.mean(vecs, axis=0)


X_train_w2v = np.vstack([w2v_sentence_vec(s) for s in X_train_text])
X_test_w2v = np.vstack([w2v_sentence_vec(s) for s in X_test_text])
X_train_w2v.shape, X_test_w2v.shape

In [ ]:
w2v_clf, y_pred_w2v, results['Word2Vec'] = train_model(
    X_train_w2v, X_test_w2v, y_train, y_test, "Word2Vec (avg embedding)"
)
results

In [ ]:
# show similar words
for w in ['good', 'bad', 'amazing', 'boring']:
    if w in w2v_model.wv:
        print(f"\nMost similar to '{w}':")
        print(w2v_model.wv.most_similar(w, topn=5))

# t-SNE visualization for a few sentiment words
words_to_plot = ['good', 'great', 'excellent', 'bad', 'terrible', 'awful', 'boring', 'amazing', 'movie', 'film']
valid_words = [w for w in words_to_plot if w in w2v_model.wv]

if valid_words:
    vecs = np.array([w2v_model.wv[w] for w in valid_words])
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(valid_words) - 1))
    reduced = tsne.fit_transform(vecs)

    plt.figure(figsize=(6, 5))
    plt.scatter(reduced[:, 0], reduced[:, 1])
    for i, w in enumerate(valid_words):
        plt.annotate(w, (reduced[i, 0], reduced[i, 1]))
    plt.title("Word2Vec Sentiment Word Clusters (t-SNE)")
    plt.show()
else:
    print("No valid words found for t-SNE plot.")

In [ ]:
from google.colab import files

files.upload()  # upload glove.6B.100d.txt

In [ ]:
glove_index = {}
with open("glove.6B.100d.txt", "r", encoding="utf8") as f:
    for line in tqdm(f):
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        glove_index[word] = coefs
len(glove_index)

In [ ]:
GLOVE_DIM = len(next(iter(glove_index.values())))

def glove_sentence_vec(text):
    words = text.split()
    vecs = [glove_index[w] for w in words if w in glove_index]
    if not vecs:
        return np.zeros(GLOVE_DIM)
    return np.mean(vecs, axis=0)


X_train_glove = np.vstack([glove_sentence_vec(s) for s in X_train_text])
X_test_glove = np.vstack([glove_sentence_vec(s) for s in X_test_text])
X_train_glove.shape, X_test_glove.shape

In [ ]:
glove_clf, y_pred_glove, results['GloVe'] = train_model(
    X_train_glove, X_test_glove, y_train, y_test, "GloVe (avg embedding)"
)
results

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(DEVICE)
bert_model.eval()

In [ ]:
def bert_embed(texts, batch_size=16):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT encoding"):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        input_ids = enc['input_ids'].to(DEVICE)
        att = enc['attention_mask'].to(DEVICE)
        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=att)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(cls_embeddings)
    return np.vstack(all_vecs)


X_train_bert = bert_embed(X_train_text)
X_test_bert = bert_embed(X_test_text)
X_train_bert.shape, X_test_bert.shape

In [ ]:
bert_clf, y_pred_bert, results['BERT'] = train_model(
    X_train_bert, X_test_bert, y_train, y_test, "BERT (CLS embedding)"
)
results

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
plot_confusion(y_test, y_pred_bow, "BoW")
plot_confusion(y_test, y_pred_tfidf, "TF-IDF")
plot_confusion(y_test, y_pred_ngram, "TF-IDF N-gram")
plot_confusion(y_test, y_pred_w2v, "Word2Vec")
plot_confusion(y_test, y_pred_glove, "GloVe")
plot_confusion(y_test, y_pred_bert, "BERT")

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results.keys(), results.values())
plt.xticks(rotation=45)
plt.ylabel("Accuracy")
plt.title("Representation Comparison on IMDB (Objective 1 & 5)")
plt.show()

In [ ]:
summary = {
    "Model": [],
    "Precision": [],
    "Recall": [],
    "F1 Score": [],
    "Accuracy": [],
}
preds = {
    "BoW": y_pred_bow,
    "TF-IDF": y_pred_tfidf,
    "TF-IDF N-gram": y_pred_ngram,
    "Word2Vec": y_pred_w2v,
    "GloVe": y_pred_glove,
    "BERT": y_pred_bert,
}
for name, pred in preds.items():
    summary["Model"].append(name)
    summary["Precision"].append(precision_score(y_test, pred))
    summary["Recall"].append(recall_score(y_test, pred))
    summary["F1 Score"].append(f1_score(y_test, pred))
    summary["Accuracy"].append(accuracy_score(y_test, pred))
metrics_df = pd.DataFrame(summary)
display(metrics_df)

# plot graph for each metric
metrics_melted = metrics_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(8, 5))
sns.barplot(data=metrics_melted, x="Model", y="Score", hue="Metric")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.title("Model Comparison Across Metrics (Accuracy, Precision, Recall, F1)")
plt.legend(loc="lower right")
plt.show()